# Week 7 — Wednesday: Choosing a Model — Cross-Validation and Grid Search

**DATA 202 · Calvin University**

Same children, same two blood measurements (`fGSH`, `GSSG`), same split: 111 training children, 48 test children.

Monday, we chose depth 2 by looking at the **test** children's scores.

> 🎯 **How can we choose a tree's depth without touching the test set?**

**Today's plan (~45 min):**

| Time | Part |
|---|---|
| ~8 min | 1 · One test set can't choose |
| ~9 min | 2 · Cross-validation: every child takes a turn (SLO 07B) |
| ~5 min | 3 · Choosing depth with cross-validation (SLO 07B) |
| ~10 min | 4 · Grid search: try every combination (SLO 07C) |
| ~8 min | 5 · Selecting vs. estimating |
| ~5 min | 6 · The best of many |

**Cues:** 🎯 Predict First · 💬 Question · 🗣️ Pair Talk · 📌 One split

In [ ]:
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

autism = pd.read_csv("https://cs.calvin.edu/courses/data/202/26fa/datasets/autism.csv")
kids = autism[autism["Group"] != "SIB"]                    # ASD vs. NEU
X = kids[["fGSH", "GSSG"]]
y = kids["Group"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=7)   # Monday's split

---
## 1 · One Test Set Can't Choose · ~7 min

> 🎯 **Predict First:** Monday's split said depth 2 or 3 is best. With a different random split, would the best depth be the same?

In [ ]:
rows = []
for seed in [7, 1, 13, 21, 30]:                           # Monday's split first, then four others
    X_a, X_b, y_a, y_b = train_test_split(X, y, test_size=0.3, random_state=seed)
    for depth in range(1, 6):
        model = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_a, y_a)
        rows.append({"random_state": seed, "max_depth": depth, "test accuracy": accuracy_score(y_b, model.predict(X_b))})

pd.DataFrame(rows).pivot(index="random_state", columns="max_depth", values="test accuracy").round(3)

> 📄 **Handout 1 — Five splits.** In each row, circle the best depth. Write down the lowest and highest score for depth 2. 🗣️ Which depth would you pick?

<details><summary>Answer</summary>

* Best depth: **2 or 3** in four splits, **4** in one
* Depth 2 scores from **77.1%** to **93.8%**: same tree recipe, same children
</details>

**Two problems with choosing on the test set:**

| Problem | What goes wrong |
|---|---|
| **Luck** | Which 48 children land in the test set moves the score by up to 17 points. One split is a noisy judge |
| **Peeking** | If we keep the depth with the best test score, the test set helped *build* the model. Its score is no longer about new children: it's the best of several tries, so it looks better than it is |

* Peeking has a name: **leakage**. Information from the data that scores a model leaks into building it
* Kapoor & Narayanan ([*Patterns*, 2023](https://doi.org/10.1016/j.patter.2023.100804)) found it in published work in many fields. Their [running list](https://reproducible.cs.princeton.edu/): **648 papers in 30 fields**

**The fix:** lock the test set until the very end. Do the choosing *inside* the training set, and do it **many times**.

---
## 2 · Cross-Validation: Every Child Takes a Turn (SLO 07B) · ~9 min

<img src="https://cs.calvin.edu/courses/data/202/26fa/weeks/07/images/cv_folds.png" width="720" alt="Diagram of 5-fold cross-validation. The 111 training children are cut into 5 folds of about 22. In each of 5 rounds, one fold (orange, 'validate') is held out and the other four (blue, 'fit') train the tree; the rounds score 0.870, 0.818, 0.864, 0.909 and 0.773. The cross-validation score is their mean, 0.847. The 48 test children sit in a gray 'locked' box in every round.">

**5-fold cross-validation:**

1. Cut the **training** children into 5 groups: the **folds**
2. Each round: fit on 4 folds, score on the 5th
3. Every child is scored **exactly once**, by a tree that never saw them
4. The **mean** of the 5 scores is the cross-validation score

In [ ]:
from sklearn.model_selection import cross_val_score

fold_scores = cross_val_score(DecisionTreeClassifier(max_depth=2, random_state=0), X_train, y_train, cv=5)
print("5 rounds:", fold_scores.round(3))
print("mean:    ", fold_scores.mean().round(3))

> 📄 **Handout 2 — Five folds.** In each round, shade the fold that is scored. Then compute the mean of the 5 scores.

<details><summary>Answer</summary>

* (0.870 + 0.818 + 0.864 + 0.909 + 0.773) ÷ 5 = **0.847**
* From 77.3% to 90.9% between rounds: the luck from Part 1, now **averaged out**
* 5 trees fitted; the 48 test children never touched
</details>

**Leave-one-out (LOO):** take it to the limit, with one fold per child. 111 rounds, each scoring a single child.

In [ ]:
from sklearn.model_selection import LeaveOneOut

loo_scores = cross_val_score(DecisionTreeClassifier(max_depth=2, random_state=0), X_train, y_train, cv=LeaveOneOut())
print(len(loo_scores), "rounds; children scored right:", int(loo_scores.sum()))
print("mean:", loo_scores.mean().round(3))

* Each round scores **0 or 1**: 95 of 111 children right → **0.856**
* Each tree trains on 110 children (vs. ~89 in 5-fold), but we fit **111 trees**
* LOO is common with small medical datasets, like the study behind our data (Part 6)

---
## 3 · Choosing Depth With Cross-Validation (SLO 07B) · ~5 min

One 5-fold score per depth. Error bars: how much the 5 rounds disagreed.

In [ ]:
cv_scores = []
for depth in range(1, 9):
    s = cross_val_score(DecisionTreeClassifier(max_depth=depth, random_state=0), X_train, y_train, cv=5)
    cv_scores.append({"max_depth": depth, "mean accuracy": s.mean(), "spread": s.std()})
cv_scores = pd.DataFrame(cv_scores)

fig = px.line(cv_scores, x="max_depth", y="mean accuracy", error_y="spread", markers=True,
              title="5-Fold Cross-Validation Accuracy vs. Depth (111 training children)")
fig.show()

> 📄 **Handout 3 — Pick the depth.** Circle the depth with the best cross-validation score. Label the underfitting and overfitting sides.

<details><summary>Answer</summary>

* **Best: depth 2** (0.847). Monday's choice, but this time the test set chose nothing
* **Depth 1:** 0.702 → **underfitting**
* **Depth 7 and up:** 0.784 → **overfitting**: the full tree memorizes the 4 folds it fits on
</details>

---
## 4 · Grid Search: Try Every Combination (SLO 07C) · ~10 min

A tree has more than one knob, like Monday's `max_depth` and `min_samples_leaf`.

**Grid search:** list values for each knob, cross-validate **every combination**, keep the best. (Part 3 was a grid with one knob.)

In [ ]:
from sklearn.model_selection import GridSearchCV

grid = {"max_depth": [1, 2, 3, 4, 5, 6, None],           # None = no limit
        "min_samples_leaf": [1, 2, 5, 10, 20]}

search = GridSearchCV(DecisionTreeClassifier(random_state=0), grid, cv=5)
search.fit(X_train, y_train)                             # training children only

In [ ]:
results = pd.DataFrame(search.cv_results_)
results["max_depth"] = [str(p["max_depth"]).replace("None", "no limit") for p in results["params"]]
table = results.pivot(index="param_min_samples_leaf", columns="max_depth", values="mean_test_score")
table = table[["1", "2", "3", "4", "5", "6", "no limit"]]

fig = px.imshow(table, text_auto=".3f", color_continuous_scale="Blues", aspect="auto",
                labels=dict(x="max_depth", y="min_samples_leaf", color="CV accuracy"),
                title="Grid Search: 5-Fold CV Accuracy for 35 Combinations")
fig.show()

> 📄 **Handout 4 — The grid.** Circle the best cell. How many trees did the grid search fit? Box every cell within 0.01 of the best.

In [ ]:
print("best combination:", search.best_params_)
print("best CV accuracy:", round(search.best_score_, 3))

<details><summary>Answer</summary>

* **Best:** `max_depth=6`, `min_samples_leaf=5` → **0.856**, a 10-leaf tree
* **Trees fitted:** 35 × 5 = **175** (plus one refit of the winner on all 111 children)
* **Boxed cells:** depth 3 (0.856) and depth 2 (0.847) are within **1 child** of the winner. The folds can't tell them apart
</details>

`GridSearchCV` takes the highest number, however complex the tree. For near-ties, the **one-standard-error rule** (Breiman et al., 1984): keep every combination within one *standard error* of the best, then take the **simplest**.

In [ ]:
results["leaves"] = [DecisionTreeClassifier(random_state=0, **p).fit(X_train, y_train).get_n_leaves() for p in results["params"]]

best = results.loc[results["mean_test_score"].idxmax()]
standard_error = best["std_test_score"] / 5 ** 0.5                 # 5 folds
close = results[results["mean_test_score"] >= best["mean_test_score"] - standard_error]

print("best:", round(best["mean_test_score"], 3), "| one standard error:", round(standard_error, 3))
close.sort_values("leaves")[["max_depth", "param_min_samples_leaf", "mean_test_score", "leaves"]].round(3)

* 9 combinations are within one standard error (0.019), from 4 to 10 leaves
* Simplest: **depth 2** (4 leaves)

Now, **once**, open the locked test set:

In [ ]:
chosen = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_train, y_train)
print("depth 2 (one-standard-error rule) | test:", round(accuracy_score(y_test, chosen.predict(X_test)), 3))

<details><summary>Answer</summary>

* **87.5%**, an honest score: the test set chose nothing
* For the record, the 10-leaf winner gets 79.2%. We don't get to switch after seeing that: that would be peeking
* 📌 **One split:** 48 children; the gap is 4 children
</details>

---
## 5 · Selecting vs. Estimating · ~8 min

💬 Many studies skip the test set and cross-validate **all** their data. Is that wrong?

| Cross-validation's job | Use all the data? |
|---|---|
| **Estimating:** how well will *this* recipe (depth 2) do on new children? | ✅ every child is scored by a tree that never saw them |
| **Selecting:** which depth? which knobs? | ✅ to choose. ❌ to report the winner's score: it's the **best of many** |

**Nested cross-validation** does both with all the data. An **inner** grid search picks the knobs on 4/5 of the children, and the **outer** loop scores that pick on the other 1/5.

📖 Raschka (2018), [*Model Evaluation, Model Selection, and Algorithm Selection in Machine Learning*](https://arxiv.org/abs/1811.12808)

**A test with a known answer:** shuffle the ASD/NEU labels so they're pure chance. No model can beat a coin flip (~50%). We run the Part 4 grid on all 159 children, 20 times.

> 🎯 **Predict First:** Which will claim to beat the coin flip: non-nested or nested?

In [ ]:
from sklearn.model_selection import KFold

rows = []
for trial in range(20):
    y_noise = y.sample(frac=1, random_state=trial).to_numpy(dtype=str)   # shuffled labels: no real signal left
    inner = KFold(5, shuffle=True, random_state=trial)
    outer = KFold(5, shuffle=True, random_state=trial + 100)
    noise_search = GridSearchCV(DecisionTreeClassifier(random_state=0), grid, cv=inner)

    non_nested = noise_search.fit(X, y_noise).best_score_                    # choose AND score on the same folds
    nested = cross_val_score(noise_search, X, y_noise, cv=outer).mean()     # choose inside, score outside
    rows += [{"trial": trial, "method": "non-nested", "CV accuracy": non_nested},
             {"trial": trial, "method": "nested", "CV accuracy": nested}]
noise = pd.DataFrame(rows)

fig = px.scatter(noise, x="trial", y="CV accuracy", color="method",
                 title="Shuffled Labels, All 159 Children: Non-Nested vs. Nested Cross-Validation")
fig.add_hline(y=0.5, line_dash="dash", annotation_text="coin flip")
fig.show()
noise.groupby("method")["CV accuracy"].mean().round(3)

<details><summary>Answer</summary>

* **Non-nested: 54.6%**, above the coin flip in 16 of 20 trials. It found "signal" in pure noise
* **Nested: 49.0%**, a coin flip, as it should be
* The data that **chose** the model can't also give its **honest score**
</details>

📖 Adapted from scikit-learn's [Nested versus non-nested cross-validation](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html). See also: [Cross-validation: evaluating estimator performance](https://scikit-learn.org/stable/modules/cross_validation.html)

---
## 6 · The Best of Many · ~5 min

The 2017 study behind our data searched over **which metabolites** to use:

> "To avoid over-fitting and over-stating results, leave-one-out cross validation is employed …"
>
> "Therefore, all combinations of up to six variables were evaluated for separability."
> — Howsmon et al. (2017)

* About **190,000** combinations, each scored by leave-one-out on all the children. The paper reports the **maximum** for each number of metabolites

> 💬 **Question:** Every score was cross-validated. Is the *best* of them a fair estimate for the next child in a clinic? What would be?

<details><summary>Answer</summary>

* Cross-validation was used to **select** and to **estimate** at once (Part 5). 35 tries found 54.6% in noise; 190,000 tries have far more chances to be lucky
* A fair check: children used for **no** choice at all, ideally from another clinic. All 159 came from one study in Arkansas
* This is the leakage from Part 1 (Kapoor & Narayanan): choosing features with the same data that scores them. It doesn't make the finding false. It means its best number needs a **new test**
</details>

---
### Wrap-up

1. **Test set:** locked until the end, opened **once**. Choosing with it is peeking (leakage)
2. **Cross-validation:** every training child is scored once by a model that never saw them (5-fold, or leave-one-out)
3. **Grid search:** cross-validate every combination; for near-ties, take the simplest (one-standard-error rule)
4. **Selecting ≠ estimating:** data that chose the model can't also score it. Use a locked test set or **nested** cross-validation 📌

**Practice 06 (Friday of Week 8, after Fall Break):** trees, cross-validation and grid search on your own.